# Advanced: Computing Eigenvalues and Eigenvectors
The [lecture](CHEME-5800-L14a-Lecture-LinearStateSpaceModels-Fall-2026.ipynb) reads the stability of a linear state space model from the eigenvalues of its state matrix, and [the `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen) computes those eigenvalues. In this notebook, we look at two iterative methods that compute eigenvalues and eigenvectors. Power iteration finds the eigenvalue of largest magnitude and its eigenvector, and QR iteration finds every eigenvalue and eigenvector.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Explain why eigenvalues are computed by iteration:__ The eigenvalues of a matrix are the roots of its characteristic polynomial, but computing those roots directly fails for large matrices. We'll see why numerical methods work on the matrix itself instead.
> * __Estimate the dominant eigenpair by power iteration:__ Power iteration multiplies a vector by the matrix again and again until the direction of the vector stops changing. We'll run it, check its eigenvalue estimate, and measure how the gap between the two largest eigenvalues sets its speed.
> * __Compute every eigenpair by QR iteration:__ The QR iteration method factors the matrix into an orthogonal factor and a triangular factor, then multiplies the factors in reverse order. We'll use it to compute every eigenvalue and eigenvector of a symmetric matrix and check each pair with its residual.

Let's get started!

___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package and the packages used here.

Let's set up our code environment:

In [1]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

The two methods in this notebook are [the `poweriteration(...)` function](../../../code/src/Eigen.jl) and [the `qriteration(...)` function](../../../code/src/Eigen.jl) in the course package.

___

## Task 1: Eigenvalues, Eigenvectors, and the Characteristic Polynomial
In this task, we define the eigendecomposition of a square matrix and see why we do not compute eigenvalues from the characteristic polynomial.

Suppose $\mathbf{A}\in\mathbb{R}^{n\times n}$ is a real square matrix. An __eigenvector__ $\mathbf{v}\neq\mathbf{0}$ of $\mathbf{A}$ is a vector that $\mathbf{A}$ only stretches, and the stretch factor $\lambda$ is its __eigenvalue__:

$$
\mathbf{A}\mathbf{v}_{j} = \lambda_{j}\mathbf{v}_{j}\qquad j = 1,2,\dots,n.
$$

The eigenvalues and eigenvectors can be complex. An eigenvalue and its eigenvector form an __eigenpair__. When the $n$ eigenvectors are linearly independent, we collect them as the columns of a matrix $\mathbf{V}$, and the eigenvalues on the diagonal of a matrix $\boldsymbol{\Lambda}$, which gives the __eigendecomposition__ $\mathbf{A}=\mathbf{V}\boldsymbol{\Lambda}\mathbf{V}^{-1}$.

For a real symmetric matrix, $\mathbf{A}=\mathbf{A}^{\top}$, every eigenvalue is real, and the eigenvectors can be chosen orthonormal, which gives $\mathbf{V}^{-1}=\mathbf{V}^{\top}$. A state matrix is usually not symmetric, and its eigenvalues can come in complex pairs.

### Why not the characteristic polynomial?
Moving every term of the eigenvalue equation to one side gives $(\mathbf{A}-\lambda\mathbf{I})\mathbf{v}=\mathbf{0}$, where $\mathbf{I}$ is the identity matrix. A nonzero $\mathbf{v}$ solves this system only when $\mathbf{A}-\lambda\mathbf{I}$ is singular. The eigenvalues are therefore the roots of the __characteristic polynomial__:

$$
p(\lambda) = \det\left(\mathbf{A}-\lambda\mathbf{I}\right) = 0.
$$

For a matrix with two rows, $p(\lambda)$ is a quadratic, and the quadratic formula gives both eigenvalues. For five or more rows, no formula built from roots and arithmetic gives the roots of a general polynomial ([the Abel–Ruffini theorem](https://en.wikipedia.org/wiki/Abel%E2%80%93Ruffini_theorem)).

Computing the coefficients and then the roots numerically does not help either. Tiny rounding errors in the coefficients can move the roots a long way, as [Wilkinson's polynomial](https://en.wikipedia.org/wiki/Wilkinson%27s_polynomial) shows. Numerical methods instead iterate on the matrix itself, with matrix products and factorizations.

We'll use one symmetric test matrix throughout. Let's compute its eigenpairs with [the `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen) as a reference, and check each pair with its residual $\mathbf{r}_{j}=\mathbf{A}\mathbf{v}_{j}-\lambda_{j}\mathbf{v}_{j}$, which is zero for an exact eigenpair:

In [2]:
A = [
    4.0 1.0 0.0 ;
    1.0 3.0 1.0 ;
    0.0 1.0 2.0 ;
]; # a symmetric 3×3 test matrix

reference = let

    # compute the eigendecomposition with the library solver -
    F = eigen(A); # F.values holds the eigenvalues (smallest to largest), F.vectors holds the eigenvectors as columns

    # print each eigenvalue and the length of its residual -
    for j ∈ 1:length(F.values)
        λ = F.values[j]; # the j-th eigenvalue
        v = F.vectors[:, j]; # its eigenvector
        println("λ = $(round(λ; digits = 6)),  ‖Av − λv‖ = $(norm(A*v - λ*v))")
    end

    F; # return
end;

λ = 1.267949,  ‖Av − λv‖ = 1.951593978782523e-15


λ = 3.0,  ‖Av − λv‖ = 2.220446049250313e-15
λ = 4.732051,  ‖Av − λv‖ = 4.440892098500626e-16


The eigenvalues are $3-\sqrt{3}\approx 1.268$, $3$, and $3+\sqrt{3}\approx 4.732$, and every residual is near the rounding error of 64-bit arithmetic. These are the values the two iterative methods should reproduce.

___

## Task 2: Power Iteration
In this task, we estimate the dominant eigenpair of $\mathbf{A}$ with power iteration, and measure how the gap between the two largest eigenvalues sets the number of iterations.

Order the eigenvalues by magnitude, $|\lambda_{1}|\geq|\lambda_{2}|\geq\dots\geq|\lambda_{n}|$. The __dominant eigenvalue__ $\lambda_{1}$ has the largest magnitude. Power iteration multiplies a starting vector by $\mathbf{A}$ and rescales the result to unit length, over and over:

$$
\mathbf{v}^{(k+1)} = \frac{\mathbf{A}\mathbf{v}^{(k)}}{\lVert\mathbf{A}\mathbf{v}^{(k)}\rVert_{2}}\qquad k = 0,1,2,\dots
$$

To see why this works, suppose $\mathbf{A}$ has $n$ linearly independent eigenvectors. We write the starting vector in the eigenvector basis, $\mathbf{v}^{(0)}=c_{1}\mathbf{v}_{1}+\dots+c_{n}\mathbf{v}_{n}$, and each multiplication by $\mathbf{A}$ multiplies the $j$-th term by $\lambda_{j}$:

$$
\mathbf{A}^{k}\mathbf{v}^{(0)} = \lambda_{1}^{k}\left(c_{1}\mathbf{v}_{1}+\sum_{j=2}^{n}c_{j}\left(\frac{\lambda_{j}}{\lambda_{1}}\right)^{k}\mathbf{v}_{j}\right).
$$

When $|\lambda_{1}|>|\lambda_{2}|$ and $c_{1}\neq 0$, every ratio in the sum goes to zero, and the direction of the vector approaches $\mathbf{v}_{1}$. The slowest term shrinks by the factor $|\lambda_{2}/\lambda_{1}|$ in each iteration. A ratio near one, a narrow __spectral gap__, means slow convergence.

When $\lambda_{1}$ is negative, the factor $\lambda_{1}^{k}$ flips the sign of the vector at each iteration, while its direction still settles. We therefore measure the change in the vector up to its sign.

Once the vector stops changing, the __Rayleigh quotient__ estimates the eigenvalue. It comes from taking the inner product of both sides of $\mathbf{A}\mathbf{v}\approx\hat{\lambda}_{1}\mathbf{v}$ with $\mathbf{v}$:

$$
\hat{\lambda}_{1} = \frac{(\mathbf{A}\mathbf{v})^{\top}\mathbf{v}}{\mathbf{v}^{\top}\mathbf{v}}.
$$

### Algorithm
__Initialize__: Given the matrix $\mathbf{A}\in\mathbb{R}^{n\times n}$, a nonzero starting vector $\mathbf{v}$, a tolerance $\epsilon>0$, and a maximum number of iterations $K$, set the iteration counter $k\gets 1$ and the flag $\texttt{stop}\gets\texttt{false}$.

While $\texttt{stop}$ is $\texttt{false}$ __do__:
1. __Multiply:__ Compute the next vector $\mathbf{w}\gets\mathbf{A}\mathbf{v}/\lVert\mathbf{A}\mathbf{v}\rVert_{2}$.
2. __Measure:__ Compute the change up to the sign, $\delta\gets\min\left(\lVert\mathbf{w}-\mathbf{v}\rVert_{2},\lVert\mathbf{w}+\mathbf{v}\rVert_{2}\right)^{2}$, and set $\mathbf{v}\gets\mathbf{w}$.
3. __Check:__ Take the first of these cases that holds:
    - If $\delta\leq\epsilon$, __then__ set $\texttt{stop}\gets\texttt{true}$, because the vector has converged.
    - If $k\geq K$, __then__ set $\texttt{stop}\gets\texttt{true}$, because the iteration limit is reached.
    - Otherwise, set $k\gets k+1$.

__Output__: The eigenvector estimate $\mathbf{v}$ and the eigenvalue estimate $\hat{\lambda}_{1}$ from the Rayleigh quotient.

[The `poweriteration(...)` function](../../../code/src/Eigen.jl) carries out these steps. Its keyword arguments `maxiter` and `ϵ` set $K$ and $\epsilon$, and it prints the number of iterations it used. Let's run it on $\mathbf{A}$ from a vector of ones, and compare the eigenvalue estimate and its residual with the reference:

In [3]:
dominant = let

    # initialize -
    v₀ = ones(3); # the starting vector
    maxiter = 500; # the most iterations, K
    ϵ = 1e-14; # the tolerance on the squared change of the vector

    # run power iteration: result.vector is the eigenvector estimate, result.value the eigenvalue estimate -
    result = poweriteration(A, v₀; maxiter = maxiter, ϵ = ϵ);
    λ̂ = result.value;
    v̂ = result.vector;

    # the reference dominant eigenvalue: the one of largest magnitude -
    λ₁ = reference.values[argmax(abs.(reference.values))];

    # print the estimate, the reference, and the length of the residual -
    println("power iteration: λ̂ = $(λ̂),  ‖Av̂ − λ̂v̂‖ = $(norm(A*v̂ - λ̂*v̂))")
    println("eigen(...):      λ₁ = $(λ₁)")

    # checks -
    @test isapprox(λ̂, λ₁; atol = 1e-8)
    @test norm(A*v̂ - λ̂*v̂) < 1e-5

    result; # return
end;

Converged in 32 iterations
power iteration: λ̂ = 4.732050807568828,  ‖Av̂ − λ̂v̂‖ = 2.940161864439526e-7
eigen(...):      λ₁ = 4.732050807568878


The estimate matches the dominant eigenvalue $3+\sqrt{3}$ after 32 iterations. For this matrix the ratio is $|\lambda_{2}/\lambda_{1}| = 3/(3+\sqrt{3})\approx 0.63$, and each iteration multiplies the remaining error by about that factor.

### Spectral Gap and Convergence Speed
The ratio $|\lambda_{2}/\lambda_{1}|$ sets how fast power iteration converges. Let's compare two diagonal matrices with the same dominant eigenvalue, $5$. The first has $\lambda_{2}=1$, a ratio of $0.2$, and the second has $\lambda_{2}=4.9$, a ratio of $0.98$:

In [4]:
let

    # initialize -
    v₀ = ones(3); # the starting vector
    maxiter = 5000; # the most iterations, K
    ϵ = 1e-12; # the tolerance on the squared change of the vector
    wide = diagm([5.0, 1.0, 0.5]); # a wide gap: |λ₂/λ₁| = 0.2
    narrow = diagm([5.0, 4.9, 0.5]); # a narrow gap: |λ₂/λ₁| = 0.98

    # run power iteration on both matrices; each call prints its number of iterations -
    print("wide gap:   ")
    fast = poweriteration(wide, v₀; maxiter = maxiter, ϵ = ϵ);
    print("narrow gap: ")
    slow = poweriteration(narrow, v₀; maxiter = maxiter, ϵ = ϵ);

    # checks -
    @test isapprox(fast.value, 5.0; atol = 1e-6)
    @test isapprox(slow.value, 5.0; atol = 1e-3)
end;

wide gap:   Converged in 10 iterations
narrow gap: Converged in 492 iterations


The wide gap needs 10 iterations, and the narrow gap needs 492. A ratio near one shrinks the error very little in each iteration.

___

## Task 3: QR Iteration
In this task, we compute every eigenpair of $\mathbf{A}$ with QR iteration, and check each pair with its residual.

The __QR decomposition__ factors a square matrix as $\mathbf{A}=\mathbf{Q}\mathbf{R}$, where $\mathbf{Q}$ is orthogonal ($\mathbf{Q}^{\top}\mathbf{Q}=\mathbf{I}$) and $\mathbf{R}$ is upper triangular. We compute it with [the `qr(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.qr). Each QR iteration factors the current matrix and multiplies the factors in reverse order:

$$
\mathbf{A}_{k}=\mathbf{Q}_{k}\mathbf{R}_{k},\qquad
\mathbf{A}_{k+1}=\mathbf{R}_{k}\mathbf{Q}_{k}=\mathbf{Q}_{k}^{\top}\mathbf{A}_{k}\mathbf{Q}_{k}.
$$

The second form follows from $\mathbf{R}_{k}=\mathbf{Q}_{k}^{\top}\mathbf{A}_{k}$. Matrices related this way are __similar__, and similar matrices have the same eigenvalues. For a real symmetric matrix whose eigenvalues have distinct magnitudes, the iterates approach a diagonal matrix with the eigenvalues on the diagonal.

For each eigenvalue estimate $\lambda_{j}$, the eigenvector solves the homogeneous system $(\mathbf{A}-\lambda_{j}\mathbf{I})\mathbf{v}_{j}=\mathbf{0}$. Row reduction of $\mathbf{A}-\lambda_{j}\mathbf{I}$ finds such a vector. Because $\lambda_{j}$ is only an estimate, row reduction can find no free column, and the method then uses the right singular vector of the smallest singular value, the unit vector that $\mathbf{A}-\lambda_{j}\mathbf{I}$ shrinks the most.

> __Symmetric matrices only:__ This basic QR iteration assumes real eigenvalues. For a matrix with a complex pair of eigenvalues, such as many state matrices, each pair leaves a 2×2 block on the diagonal of the iterates. The general solver behind [the `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen) uses a shifted QR iteration that handles these blocks.

### Algorithm
__Initialize__: Given a real symmetric matrix $\mathbf{A}\in\mathbb{R}^{n\times n}$, a tolerance $\epsilon>0$, and a maximum number of iterations $K$, set $\mathbf{A}_{1}\gets\mathbf{A}$. Set the starting estimates $\boldsymbol{\lambda}$ to the diagonal of the $\mathbf{Q}$ factor of $\mathbf{A}$, which only the first comparison uses.

For $k = 1,2,\dots,K$ __do__:
1. __Factor:__ Compute the QR decomposition $\mathbf{A}_{k}=\mathbf{Q}_{k}\mathbf{R}_{k}$.
2. __Update:__ Set $\mathbf{A}_{k+1}\gets\mathbf{Q}_{k}^{\top}\mathbf{A}_{k}\mathbf{Q}_{k}$ and the new estimates $\boldsymbol{\lambda}^{\prime}\gets\operatorname{diag}(\mathbf{A}_{k+1})$.
3. __Check:__ Compute the change $\delta\gets\lVert\boldsymbol{\lambda}^{\prime}-\boldsymbol{\lambda}\rVert_{2}$ and set $\boldsymbol{\lambda}\gets\boldsymbol{\lambda}^{\prime}$. If $\delta<\epsilon$, __then__ stop the loop.

__Sort:__ Sort the eigenvalue estimates from smallest to largest.

For each $j = 1,2,\dots,n$ __do__:
1. __Null space:__ Find a vector $\mathbf{u}$ with $(\mathbf{A}-\lambda_{j}\mathbf{I})\mathbf{u}=\mathbf{0}$ by row reduction. If row reduction finds no free column, __then__ set $\mathbf{u}$ to the right singular vector of the smallest singular value of $\mathbf{A}-\lambda_{j}\mathbf{I}$.
2. __Scale:__ Set the eigenvector $\mathbf{v}_{j}\gets\mathbf{u}/\lVert\mathbf{u}\rVert_{2}$.

__Output__: The eigenvalues $\lambda_{1},\dots,\lambda_{n}$, from smallest to largest, and the unit eigenvectors $\mathbf{v}_{1},\dots,\mathbf{v}_{n}$.

[The `qriteration(...)` function](../../../code/src/Eigen.jl) carries out these steps. Its keyword arguments `maxiter` and `tolerance` set $K$ and $\epsilon$. It returns the eigenvalues as a vector and the eigenvectors in a dictionary keyed by the index $j$.

Let's compute every eigenpair of $\mathbf{A}$, and compare each eigenvalue and residual with the reference:

In [5]:
let

    # initialize -
    maxiter = 200; # the most iterations, K
    tolerance = 1e-9; # the tolerance on the change in the diagonal, ϵ

    # run QR iteration: λ holds the eigenvalues (smallest to largest), V[j] the eigenvector of λ[j] -
    (λ, V) = qriteration(A; maxiter = maxiter, tolerance = tolerance);

    # print each eigenvalue, the reference eigenvalue, and the length of the residual -
    for j ∈ 1:length(λ)
        residual = norm(A*V[j] - λ[j]*V[j]); # ‖Av − λv‖ for the j-th eigenpair
        println("λ[$(j)] = $(round(λ[j]; digits = 10)),  eigen(...) = $(round(reference.values[j]; digits = 10)),  ‖Av − λv‖ = $(residual)")
    end

    # checks -
    for j ∈ 1:length(λ)
        @test isapprox(λ[j], reference.values[j]; atol = 1e-8)
        @test norm(A*V[j] - λ[j]*V[j]) < 1e-5
    end
end;

λ[1] = 1.2679491924,  eigen(...) = 1.2679491924,  ‖Av − λv‖ = 5.578801654593729e-16


λ[2] = 3.0000000003,  eigen(...) = 3.0,  ‖Av − λv‖ = 5.077560594202168e-10
λ[3] = 4.7320508073,  eigen(...) = 4.7320508076,  ‖Av − λv‖ = 2.9314615728757197e-10


Every eigenvalue matches the reference to about nine digits, and every residual is below $10^{-9}$.

___

## Summary
In this notebook, we computed the eigenvalues and eigenvectors of a symmetric matrix with two iterative methods, and checked each result against [the `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen).

> __Key Takeaways:__
>
> * __Eigenvalues are computed by iteration:__ We saw that the eigenvalues are the roots of the characteristic polynomial, but that no formula gives those roots for large matrices, and rounding errors in the coefficients move them. We therefore compute eigenvalues by iterating on the matrix itself.
> * __Power iteration finds the dominant eigenpair:__ We multiplied a vector by the matrix until its direction stopped changing, and estimated the eigenvalue with the Rayleigh quotient. We also saw that a narrow gap between the two largest eigenvalue magnitudes makes the method slow.
> * __The QR iteration method finds every eigenpair:__ We factored the matrix, multiplied the factors in reverse order, and read the eigenvalues from the diagonal of the result. We then solved a homogeneous linear system for each eigenvector and checked every eigenpair with its residual.

___